In [18]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_stx"


In [19]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 238 firm-variable mappings across 130 firms.


firm_id,variable,notes,final_choice,num_candidates
8TRA.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A because 'Distribution expenses' is a clear selling/overhead component present in column A and should be included. Chose component rows only; excluded restructuring-specific row as non-core and avoided totals/subtotals. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Distribution expenses,Income Statement::Distribution expenses; Income Statement::Administrative Expenses; Income Statement::Other operating expenses,3
AAK.ST,BE,"Preferred owner equity excluding non-controlling interests. Candidate list missed the more appropriate 'Shareholders equity' row, so manual review is required. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Shareholders equity",Balance Sheet::Shareholders equity,2
AAK.ST,COGS,"Candidate list was empty. Selected the apparent cost-of-sales components from the income statement. Manual review needed because no explicit single COGS/Cost of sales row is provided, so this relies on component mapping.",Income Statement::Raw materials and consumables; Income Statement::Change in inventories of finished goods and work in progress; Income Statement::Goods for resale,0
AAK.ST,XINT,Used 'Interest expenses' as preferred line and added 'Financial expense' as fallback because 2025 appears to use that broader label while the candidate list omits it. Manual review needed due to outside-candidate selection. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Financial expense,Income Statement::Interest expenses; Income Statement::Financial expense,5
AAK.ST,XSGA_COMPONENTS,"No explicit SG&A subtotal is present. Used overhead-like component rows instead. Manual review needed because the candidate list misses major overhead components and there are duplicated 'Other operating expenses' labels in the sheet. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Other external expenses, [Income Statement] Cost for remuneration to employees",Income Statement::Other external expenses; Income Statement::Cost for remuneration to employees; Income Statement::Other operating expenses,1
ABB.ST,BE,"No BE candidates were provided. Selected 'Total ABB stockholders equity' from the balance sheet because it appears to be parent/owner equity and excludes noncontrolling interests, which avoids double counting with MIB.",Balance Sheet::Total ABB stockholders equity,0
ABB.ST,XINT,"Candidate list misses the best interest-expense concept. 'Interest and other finance expense' is the clearest reported finance cost line, but this exact label appears twice in the sheet (one populated, one mostly empty), creating duplicate-label ambiguity. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Interest and other finance expense",Income Statement::Interest and other finance expense,3
ACADE.ST,XSGA_COMPONENTS,"No explicit SG&A subtotal row is present. Candidate list includes Personnel expenses, but it appears incomplete because Other external expenses is also a likely SG&A/overhead component from the A-column preview. Chose both component rows and excluded depreciation/amortization, impairments, finance items, and totals. Manual review recommended to confirm whether Other external expenses is fully SG&A versus partly direct operating cost. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Other external expenses",Income Statement::Personnel expenses; Income Statement::Other external expenses,2
ACAST.ST,XSGA_COMPONENTS,"Candidate list is empty. No explicit single SG&A row is present, so selected the best overhead component rows from the Income Statement. Excluded 'Product development costs' as R&D, excluded totals/balancing rows, and excluded one-off acquisition/relisting/CEO-change items.",Income Statement::Sales and marketing costs; Income Statement::Administration expenses,0
ACELP.

In [16]:
df_review = pd.DataFrame(rows)

df_xsga = (
    df_review[df_review["variable"] == "XSGA_COMPONENTS"]
    .sort_values(["firm_id"])
    .reset_index(drop=True)
)

print(f"XSGA_COMPONENTS manual reviews: {len(df_xsga)} rows across {df_xsga['firm_id'].nunique() if len(df_xsga) else 0} firms.")
display(df_xsga)

XSGA_COMPONENTS manual reviews: 91 rows across 91 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,8TRA.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A bec...,Income Statement::Distribution expenses; Incom...,3
1,AAK.ST,XSGA_COMPONENTS,No explicit SG&A subtotal is present. Used ove...,Income Statement::Other external expenses; Inc...,1
2,ACADE.ST,XSGA_COMPONENTS,No explicit SG&A subtotal row is present. Cand...,Income Statement::Personnel expenses; Income S...,2
3,ACAST.ST,XSGA_COMPONENTS,Candidate list is empty. No explicit single SG...,Income Statement::Sales and marketing costs; I...,0
4,ACELP.ST,XSGA_COMPONENTS,"Candidate list was empty, so rows were chosen ...",Income Statement::Administrative costs; Income...,4
...,...,...,...,...,...
86,MEABb.ST,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A. No...,Income Statement::Personnel Expenses; Income S...,1
87,MEKO.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel expenses; Income S...,2
88,MOMENT.ST,XSGA_COMPONENTS,No explicit SG&A total row is present. Used ov...,Income Statement::Personnel costs; Income Stat...,2
89,MSABb.ST,XSGA_COMPONENTS,Used rows outside the candidate list because t...,Income Statement::Selling and Administration E...,1
